# Wan 2.2 T2V-A14B Video Generation with Optional LoRAs

Generate video directly from text with the official `Wan-AI/Wan2.2-T2V-A14B-Diffusers` model. No starting image is required. This notebook loads `WanPipeline`, generates variations sequentially, and saves MP4s with generation metadata by prompt.

Assets belong under `Models/Model_Families/WAN/`. The application catalog entry is in `checkpoints/wan2_2_t2v/`; this notebook uses its own editable settings rather than reading catalog overrides. Checkpoint and LoRA paths resolve from this family's `checkpoints/` and `loras/` directories. `.env`, relative `CACHE_LOCATION`, and output paths resolve from the project root. Run the kernel anywhere inside this project.

The base model is selected with LoRAs disabled. Only use adapters explicitly compatible with **WAN 2.2 T2V-A14B** and the selected high-noise or low-noise expert. The existing I2V adapter presets are not selected here.


### Install Packages

Use the `huggingface` kernel with your Thor-compatible CUDA/PyTorch environment. This cell installs the inference and video-export dependencies; it does not explicitly install PyTorch, torchvision, xformers, or FlashAttention. After installation, `%pip check` checks the environment for dependency conflicts. Restart the kernel before running the pipeline.

MP4 export requires FFmpeg. `imageio-ffmpeg` can use a bundled executable or a system installation; on ARM64, install system `ffmpeg` if an executable is unavailable.

This shared environment uses Transformers 5 and Hugging Face Hub 1.32 or newer. Capping Transformers below 5 forces an older Hub version that conflicts with `datasets 5.1.0`. The bounds below let pip restore the compatible stack without downgrading `datasets`.


In [1]:
# Keep the CUDA-enabled PyTorch build supplied for your JetPack/Thor environment.
from importlib.metadata import version
print(f"Installed PyTorch: {version('torch')}")

%pip install --upgrade "diffusers>=0.42,<1" "transformers>=5,<6" "accelerate>=1.2" "peft>=0.17" "huggingface_hub>=1.32,<3" safetensors python-dotenv sentencepiece ftfy pillow numpy imageio imageio-ffmpeg

# Check all installed packages, including datasets in the shared environment.
%pip check


Installed PyTorch: 2.14.1+cu132
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 39.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 51.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 48.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 76.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 MB 63.6 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: sentencepiece
    Found existing installation: sentencepiece 0.2.2
    Uninstalling sentencepiece-0.2.2:
      Successfully uninstalled sentencepiece-0.2.2
  Attempting uninstall: imageio-ffmpeg
    Found existing installation: imageio-ffmpeg 0.7.0
    Uninstalling imageio-ffmpeg-0.7.0:
      Successfully uninstalled imageio-ffmpeg-0.7.0
  Attempting uninstall: imageiom━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/7 [imageio-ffmpeg]
    Found existing installation: ImageIO 2.38.1━━━━━━━━━━━━━━━ 1/7 [imageio-ffmpeg]
    Uninstalling ImageIO-2.38.

## Run Pipeline

1. Describe the scene, subjects, motion, and camera movement in `prompts`.
2. Edit the settings and run the settings and pipeline cells. No source image is used. Variations run sequentially.
3. Find MP4s and JSON metadata in `Videos/<prompt>/`. `saved_paths` contains the latest run's MP4 paths. No conditioning PNG is created.

Defaults use 832×480 output, 81 frames at 16 fps (about 5.06 seconds), 40 steps, high-noise guidance 4.0, and low-noise guidance 3.0. The steps and guidance follow the official T2V Diffusers example. Width and height are exact output dimensions and must be multiples of 16; frame counts must be `4*n + 1`, at least 5. Raising resolution or frame count increases memory and generation time.

The first pipeline run downloads the complete T2V Diffusers model, including **both** A14B experts. The text encoder and experts use BF16; the VAE uses FP32 with optional tiling. Components stay on CUDA, which shares Thor's system memory. Reruns reuse the loaded pipeline. `CACHE_LOCATION` (with a `hub/` subdirectory) and optional `HUGGINGFACE_TOKEN` or `HF_TOKEN` are read from the project `.env`. A local override must be a complete T2V `WanPipeline` Diffusers directory, not an I2V pipeline or a single transformer file. The catalog folder alone contains metadata, not a downloaded pipeline.

Optional LoRAs load separately into the high-noise and low-noise experts. Use matching **T2V-A14B** files supported by Diffusers. The base-model sampling defaults do not automatically adjust for acceleration/distillation adapters.

References: [official T2V model and Diffusers example](https://huggingface.co/Wan-AI/Wan2.2-T2V-A14B-Diffusers), [Wan pipeline API](https://huggingface.co/docs/diffusers/api/pipelines/wan), [Wan LoRA loader](https://huggingface.co/docs/diffusers/api/loaders/lora#diffusers.loaders.WanLoraLoaderMixin).


In [2]:
# Edit these settings, then run this cell and the pipeline cell below.
from pathlib import Path

# Find the project from either its root or a notebook's nested working directory.
project_root = next(
    (parent for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (parent / "Models" / "Model_Families").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Start the notebook kernel inside the Image_Generator project.")
model_family = "WAN"
family_dir = project_root / "Models" / "Model_Families" / model_family
checkpoint_root = family_dir / "checkpoints"
lora_root = family_dir / "loras"

def resolve_path(value, relative_to):
    """Resolve relative paths against an explicit root; preserve absolute paths."""
    path = Path(value).expanduser()
    return (path if path.is_absolute() else relative_to / path).resolve()

# Describe the full scene and motion; no source image is used.
prompts = [
    "A cinematic view of a mountain lake at sunrise. A gentle breeze ripples the water as the camera slowly moves forward.",
]

# Diffusers loads both experts, the text encoder, VAE, tokenizer, and scheduler.
model_id = "Wan-AI/Wan2.2-T2V-A14B-Diffusers"
# Optional complete local Diffusers model directory under this family's checkpoints/,
# or a local path. None = download/reuse the official model in the Hugging Face cache.
# A single SDXL checkpoint or a ComfyUI transformer file is not a complete Wan pipeline.
checkpoint_dir = None

height = 480                  # Exact output height; use 720 with width 1280 for 720p.
width = 832                   # Exact output width; both dimensions must be multiples of 16.
num_frames = 81                # Must be 4*n + 1; 81 frames at 16 fps is about 5.06 seconds.
fps = 16                      # MP4 playback rate; changing this does not generate more frames.
num_inference_steps = 40      # Standard base-model sampling, not a four-step distilled workflow.
guidance_scale = 4.0          # High-noise expert; > 1 enables negative prompting.
guidance_scale_2 = 3.0        # Low-noise expert; both guidance scales must enable/disable CFG together.
negative_prompt = "blurry, low quality, distorted, flickering, subtitles, watermark"
num_videos_per_prompt = 1     # Variations generated sequentially to limit peak memory.
seed = None                   # None = random; an integer = repeatable seed sequence.
enable_vae_tiling = True      # Reduce memory needed when encoding/decoding video.

# Optional Wan 2.2 T2V-A14B-compatible LoRAs; disabled for the base model.
# Use paths relative to this family's loras/ folder, or absolute paths. Set either expert's file
# to None to leave that expert unchanged. I2V-A14B, SDXL, FLUX, and TI2V-5B adapters
# are not assumed compatible. Select adapters explicitly trained for T2V and the target expert.
# Rerun both cells after changing the model or LoRA settings.
Use_LoRA = False
LoRA_High_Noise_Weights = None
LoRA_Low_Noise_Weights = None
LoRA_Scale = 0.8
# Add any trigger words required by your selected LoRA to the prompts.

# Relative output paths are anchored at the project root.
output_dir = "Videos"
show_videos = True            # Display saved MP4s inline.


In [3]:
import gc
import hashlib
import json
import math
import os
from pathlib import Path
import re
import secrets
from time import perf_counter
from uuid import uuid4

from dotenv import load_dotenv
from IPython.display import Video, display
import torch
import diffusers
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video
import imageio_ffmpeg

# Validate settings and local inputs before downloading weights or releasing a model.
if not isinstance(prompts, (list, tuple)) or not prompts:
    raise ValueError("prompts must be a nonempty list of strings.")
if any(not isinstance(prompt, str) or not prompt.strip() for prompt in prompts):
    raise ValueError("Every prompt must be a nonempty string.")
unique_prompts = list(dict.fromkeys(prompts))
if not isinstance(model_id, str) or not model_id.strip():
    raise ValueError("model_id must be a nonempty Hugging Face model ID.")
for name, value in {"height": height, "width": width}.items():
    if type(value) is not int or value <= 0 or value % 16:
        raise ValueError(f"{name} must be a positive integer divisible by 16.")
for name, value in {
    "num_inference_steps": num_inference_steps,
    "num_videos_per_prompt": num_videos_per_prompt,
}.items():
    if type(value) is not int or value <= 0:
        raise ValueError(f"{name} must be a positive integer.")
if type(num_frames) is not int or num_frames < 5 or (num_frames - 1) % 4:
    raise ValueError("num_frames must be 4*n + 1 and at least 5 (for example, 49 or 81).")
if seed is not None and (type(seed) is not int or not 0 <= seed < 2**63):
    raise ValueError("seed must be None or an integer between 0 and 2**63 - 1.")
for name, value in {
    "guidance_scale": guidance_scale,
    "guidance_scale_2": guidance_scale_2,
    "LoRA_Scale": LoRA_Scale,
    "fps": fps,
}.items():
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value < 0:
        raise ValueError(f"{name} must be a finite number >= 0.")
if fps == 0:
    raise ValueError("fps must be greater than zero.")
# Diffusers decides whether to encode a negative prompt from the high-noise scale.
if (guidance_scale > 1) != (guidance_scale_2 > 1):
    raise ValueError("Both guidance scales must be > 1 for CFG, or both <= 1 to disable it.")
if not isinstance(negative_prompt, str):
    raise ValueError("negative_prompt must be a string; use an empty string to leave it blank.")
for name, value in {
    "Use_LoRA": Use_LoRA,
    "enable_vae_tiling": enable_vae_tiling,
    "show_videos": show_videos,
}.items():
    if type(value) is not bool:
        raise ValueError(f"{name} must be True or False.")

checkpoint_path = None
checkpoint_key = None
if checkpoint_dir is not None:
    if not isinstance(checkpoint_dir, (str, Path)) or not str(checkpoint_dir).strip():
        raise ValueError("checkpoint_dir must be a local Diffusers directory, or None.")
    checkpoint_path = resolve_path(checkpoint_dir, checkpoint_root)
    if not checkpoint_path.is_dir() or not (checkpoint_path / "model_index.json").is_file():
        raise ValueError(f"Expected a complete Diffusers model directory with model_index.json: {checkpoint_path}")
    model_index = json.loads((checkpoint_path / "model_index.json").read_text(encoding="utf-8"))
    if not isinstance(model_index, dict) or model_index.get("_class_name") != "WanPipeline":
        raise ValueError("checkpoint_dir must contain a T2V WanPipeline, not an image-to-video pipeline.")
    # Also reload if local config or weights are replaced at the same path.
    checkpoint_key = (
        str(checkpoint_path),
        tuple(
            (str(path.relative_to(checkpoint_path)), path.stat().st_size, path.stat().st_mtime_ns)
            for path in sorted(checkpoint_path.rglob("*"))
            if path.is_file() and path.suffix in {".json", ".safetensors"}
        ),
    )

lora_paths = {}
lora_key = None
if Use_LoRA:
    for expert, weights in {
        "high_noise": LoRA_High_Noise_Weights,
        "low_noise": LoRA_Low_Noise_Weights,
    }.items():
        if weights is None:
            continue
        if not isinstance(weights, (str, Path)) or not str(weights).strip():
            raise ValueError(f"Select a local {expert} LoRA .safetensors file, or None.")
        path = resolve_path(weights, lora_root)
        if not path.is_file():
            raise FileNotFoundError(f"LoRA file not found: {path}")
        if path.suffix.lower() != ".safetensors":
            raise ValueError("Select local .safetensors LoRA files.")
        lora_paths[expert] = path
    if not lora_paths:
        raise ValueError("Use_LoRA is True; select at least one high-noise or low-noise LoRA file.")
    lora_key = tuple(
        (expert, str(path), path.stat().st_size, path.stat().st_mtime_ns)
        for expert, path in lora_paths.items()
    )

# Check MP4 encoding support before loading the large model.
imageio_ffmpeg.get_ffmpeg_exe()
load_dotenv(project_root / ".env")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a kernel with the Thor-compatible CUDA build of PyTorch.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This notebook requires CUDA BF16 support for Wan 2.2 T2V-A14B.")

device = torch.device("cuda:0")
dtype = torch.bfloat16
gib = 1024**3
free_bytes, total_bytes = torch.cuda.mem_get_info(device)
print(f"GPU: {torch.cuda.get_device_name(device)} | CUDA: {torch.version.cuda}")
print(f"CUDA memory: {free_bytes / gib:.1f} GiB free / {total_bytes / gib:.1f} GiB total")

# CACHE_LOCATION is the HF_HOME root; retain its hub/ layout from the other notebooks.
cache_root = resolve_path(os.getenv("CACHE_LOCATION") or "Models", project_root)
cache_dir = cache_root / "hub"
cache_dir.mkdir(parents=True, exist_ok=True)
model_source = str(checkpoint_path) if checkpoint_path else model_id
load_options = {
    "cache_dir": str(cache_dir),
    "token": os.getenv("HUGGINGFACE_TOKEN") or os.getenv("HF_TOKEN") or None,
    "local_files_only": checkpoint_path is not None,
    "use_safetensors": True,
    "low_cpu_mem_usage": True,
}

# Keep components on CUDA: CPU offloading also consumes Thor's shared system RAM.
# Reuse the T2V pipeline when only prompts or generation settings change.
pipeline_config = (model_source, checkpoint_key, str(cache_dir.resolve()), dtype, str(device))
if (
    "pipe" not in globals()
    or not isinstance(pipe, WanPipeline)
    or globals().get("_loaded_wan_t2v_config") != pipeline_config
):
    if "pipe" in globals():
        del pipe
    gc.collect()
    torch.cuda.empty_cache()
    # FP32 VAE follows Diffusers guidance for stable Wan encoding/decoding.
    pipe = WanPipeline.from_pretrained(
        model_source,
        vae=AutoencoderKLWan.from_pretrained(
            model_source, subfolder="vae", torch_dtype=torch.float32, **load_options
        ),
        torch_dtype=dtype,
        **load_options,
    ).to(device)
    _loaded_wan_t2v_config = pipeline_config
else:
    print("Reusing the loaded pipeline.")
if pipe.transformer is None or pipe.transformer_2 is None:
    raise ValueError("Select a Wan 2.2 T2V-A14B Diffusers pipeline with both high-noise and low-noise experts.")
boundary = pipe.config.boundary_ratio
if (type(boundary) not in (int, float) or not math.isfinite(boundary)
        or not 0 < boundary < 1):
    raise ValueError("The T2V pipeline requires an expert boundary_ratio between 0 and 1.")
# T2V-A14B denoises 16 latent channels; I2V experts expect extra image channels.
for name in ("transformer", "transformer_2"):
    config = getattr(pipe, name).config
    if config.in_channels != 16 or config.out_channels != 16:
        raise ValueError(f"{name} must be a WAN 2.2 T2V-A14B expert with 16 input/output channels.")
if enable_vae_tiling:
    pipe.vae.enable_tiling()
else:
    pipe.vae.disable_tiling()
print(f"Model: {model_source} | Expert boundary: {pipe.config.boundary_ratio}")

# Load separate LoRAs into the matching experts; never fuse into the base weights.
if getattr(pipe, "_notebook_lora_key", None) != lora_key:
    pipe.unload_lora_weights()
    pipe._notebook_lora_key = None
    try:
        for expert, path in lora_paths.items():
            pipe.load_lora_weights(
                str(path.parent),
                weight_name=path.name,
                adapter_name="notebook_lora",
                load_into_transformer_2=(expert == "low_noise"),
                local_files_only=True,
            )
    except Exception as exc:
        pipe.unload_lora_weights()
        raise RuntimeError("Could not load the LoRAs. Check Wan 2.2 T2V-A14B and expert compatibility.") from exc
    pipe._notebook_lora_key = lora_key
if Use_LoRA:
    pipe.set_adapters("notebook_lora", adapter_weights=float(LoRA_Scale))
    print(f"LoRAs: {lora_paths} | Strength: {LoRA_Scale}")
else:
    print("LoRA disabled; using the base model.")
lora_settings = {
    "enabled": Use_LoRA,
    "high_noise_weights": str(lora_paths["high_noise"]) if "high_noise" in lora_paths else None,
    "low_noise_weights": str(lora_paths["low_noise"]) if "low_noise" in lora_paths else None,
    "scale": float(LoRA_Scale) if Use_LoRA else 0.0,
}

for name in ("transformer", "transformer_2", "text_encoder", "vae"):
    component = getattr(pipe, name)
    print(f"{name}: {component.device}, {component.dtype}")
del component

# Validate the requested canvas and duration against the loaded VAE and transformer.
patch = pipe.transformer.config.patch_size
if len(patch) != 3 or any(type(value) is not int or value <= 0 for value in patch):
    raise ValueError("The T2V transformer has an invalid patch size.")
spatial = pipe.vae_scale_factor_spatial
temporal = pipe.vae_scale_factor_temporal
if type(spatial) is not int or spatial <= 0 or type(temporal) is not int or temporal <= 0:
    raise ValueError("The T2V VAE must have positive integer spatial and temporal scale factors.")
height_multiple = math.lcm(16, spatial * patch[1])
width_multiple = math.lcm(16, spatial * patch[2])
video_height, video_width = height, width
if video_height % height_multiple or video_width % width_multiple:
    raise ValueError(f"This model requires height/width multiples of {height_multiple}/{width_multiple}.")
if (num_frames - 1) % temporal or ((num_frames - 1) // temporal + 1) % patch[0]:
    raise ValueError("num_frames does not match the loaded VAE/transformer temporal alignment.")
print(f"Text-to-video canvas: {video_width}x{video_height}")
print(f"Video: {num_frames} frames at {fps} fps ({num_frames / fps:.2f}s)")

output_root = resolve_path(output_dir, project_root)
output_root.mkdir(parents=True, exist_ok=True)
saved_paths = []
torch.cuda.reset_peak_memory_stats(device)

# Process prompts and variations sequentially; never batch several A14B videos in RAM.
with torch.inference_mode():
    for prompt_index, prompt in enumerate(unique_prompts):
        slug = re.sub(r"[^a-z0-9]+", "-", prompt.lower()).strip("-")[:60].rstrip("-") or "prompt"
        prompt_hash = hashlib.sha256(prompt.encode("utf-8")).hexdigest()[:16]
        prompt_dir = output_root / f"{slug}-{prompt_hash}"
        prompt_dir.mkdir(parents=True, exist_ok=True)
        (prompt_dir / "prompt.txt").write_text(prompt, encoding="utf-8")

        for video_index in range(num_videos_per_prompt):
            video_seed = (
                secrets.randbelow(2**63) if seed is None
                else (seed + prompt_index * num_videos_per_prompt + video_index) % (2**63)
            )
            generator = torch.Generator(device=device).manual_seed(video_seed)
            generation_settings = {
                "height": video_height,
                "width": video_width,
                "num_frames": num_frames,
                "num_inference_steps": num_inference_steps,
                "guidance_scale": guidance_scale,
                "guidance_scale_2": guidance_scale_2,
                "negative_prompt": negative_prompt,
                "num_videos_per_prompt": 1,
                "output_type": "np",
            }
            print(f"Prompt {prompt_index + 1}/{len(unique_prompts)}, video {video_index + 1}/{num_videos_per_prompt}: {prompt}")
            torch.cuda.synchronize(device)
            start = perf_counter()
            frames = pipe(
                prompt=prompt, generator=generator, **generation_settings
            ).frames[0]
            torch.cuda.synchronize(device)
            elapsed = perf_counter() - start

            frame_count = len(frames)
            if frame_count != num_frames:
                raise RuntimeError(f"Expected {num_frames} decoded frames, got {frame_count}.")
            # Unique filenames preserve previous runs, including repeated seeds.
            output_path = prompt_dir / f"wan22-t2v-seed-{video_seed}-{uuid4().hex}.mp4"
            export_to_video(frames, str(output_path), fps=fps, macro_block_size=1)
            del frames  # Do not retain a previous decoded video during the next generation.
            metadata = {
                "prompt": prompt,
                "model_id": model_id,
                "checkpoint_dir": str(checkpoint_path) if checkpoint_path else None,
                "mode": "txt2vid",
                "model_variant": "wan_2_2_t2v_a14b",
                "pipeline_class": type(pipe).__name__,
                "audio_exported": False,
                "frame_count": frame_count,
                "requested_height": height,
                "requested_width": width,
                "lora": dict(lora_settings),
                "seed": video_seed,
                "dtype": str(dtype),
                "vae_dtype": str(pipe.vae.dtype),
                "enable_vae_tiling": enable_vae_tiling,
                "device": str(device),
                "torch_version": torch.__version__,
                "diffusers_version": diffusers.__version__,
                "scheduler": dict(pipe.scheduler.config),
                "boundary_ratio": pipe.config.boundary_ratio,
                "fps": fps,
                "duration_seconds": frame_count / fps,
                "generation_seconds": elapsed,
                "videos_requested_per_prompt": num_videos_per_prompt,
                **generation_settings,
            }
            output_path.with_suffix(".json").write_text(
                json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
            )
            saved_paths.append(output_path)
            print(f"Saved {output_path} | Generation: {elapsed:.1f}s")
            if show_videos:
                display(Video(filename=str(output_path), embed=True, html_attributes="controls loop"))

print(f"Saved {len(saved_paths)} videos across {len(unique_prompts)} prompt folders.")
print(f"Peak PyTorch CUDA allocation: {torch.cuda.max_memory_allocated(device) / gib:.1f} GiB")


GPU: NVIDIA Thor | CUDA: 13.2
CUDA memory: 102.2 GiB free / 122.9 GiB total


/home/malneyugnfl/anaconda3/envs/huggingface/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

/home/malneyugnfl/anaconda3/envs/huggingface/lib/python3.11/site-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


model_index.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

... (6 outputs left out)

Model: Wan-AI/Wan2.2-T2V-A14B-Diffusers | Expert boundary: 0.875
LoRA disabled; using the base model.
transformer: cuda:0, torch.bfloat16
transformer_2: cuda:0, torch.bfloat16
text_encoder: cuda:0, torch.bfloat16
vae: cuda:0, torch.float32
Text-to-video canvas: 832x480
Video: 81 frames at 16 fps (5.06s)
Prompt 1/1, video 1/1: A cinematic view of a mountain lake at sunrise. A gentle breeze ripples the water as the camera slowly moves forward.


  0%|          | 0/40 [00:00<?, ?it/s]

Saved /home/malneyugnfl/SoftwareDev/Image_Generator/Videos/a-cinematic-view-of-a-mountain-lake-at-sunrise-a-gentle-bree-72a31183b8f8b79f/wan22-t2v-seed-4194905984997583212-ecdd60c871c64e41a11a4195d4302ec3.mp4 | Generation: 3556.4s


Saved 1 videos across 1 prompt folders.
Peak PyTorch CUDA allocation: 68.1 GiB
